# 3 – Model Building
This notebook contains full Day 3 implementation including:
- Logistic Regression (from scratch)
- Logistic Regression (sklearn)
- Random Forest Classifier

Note: This notebook assumes Day 2 preprocessing is already completed and variables:
`X_train_scaled`, `X_test_scaled`, `y_train`, `y_test` are available.

# First again run train test split 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv("../Dataset/clean.csv")
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,HasMortgage,...,Education_PhD,EmploymentType_Part-time,EmploymentType_Self-employed,EmploymentType_Unemployed,MaritalStatus_Married,MaritalStatus_Single,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other
0,56,85994,50587,520,80,4,15.23,36,0.44,1,...,False,False,False,False,False,False,False,False,False,True
1,69,50432,124440,458,15,1,4.81,60,0.68,0,...,False,False,False,False,True,False,False,False,False,True
2,46,84208,129188,451,26,3,21.17,24,0.31,1,...,False,False,False,True,False,False,False,False,False,False
3,32,31713,44799,743,0,3,7.07,24,0.23,0,...,False,False,False,False,True,False,True,False,False,False
4,60,20437,9139,633,8,4,6.51,48,0.73,0,...,False,False,False,True,False,False,False,False,False,False


In [2]:
X = df.drop('Default', axis=1)
y = df['Default']

X.shape, y.shape

((255347, 24), (255347,))

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

X_train.shape, X_test.shape

((204277, 24), (51070, 24))

In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


((204277, 24), (51070, 24))

## Import Required Libraries

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


## Load Preprocessed Data
(Modify based on your environment – This is a placeholder).

In [6]:
# df = pd.read_csv("../Dataset/clean.csv")
# X = df.drop('Default', axis=1)
# y = df['Default']
# After preprocessing, you should have these variables ready:
X_train_scaled, X_test_scaled, y_train, y_test


(array([[ 0.09948564, -1.16730672,  1.6986275 , ..., -0.49975216,
         -0.50107521, -0.49917837],
        [ 0.29961984,  1.31974689, -0.86416953, ..., -0.49975216,
         -0.50107521,  2.00329193],
        [ 0.23290844,  0.4534966 , -1.70095399, ..., -0.49975216,
         -0.50107521, -0.49917837],
        ...,
        [ 1.10015666, -0.87398917, -1.38541734, ..., -0.49975216,
         -0.50107521, -0.49917837],
        [-0.43420557, -0.66288573, -0.72104648, ..., -0.49975216,
         -0.50107521,  2.00329193],
        [ 0.03277424,  0.85391829, -0.18401055, ...,  2.00099184,
         -0.50107521, -0.49917837]], shape=(204277, 24)),
 array([[ 0.69988825, -1.5799938 , -0.56955672, ...,  2.00099184,
         -0.50107521, -0.49917837],
        [ 0.36633124,  1.43613989,  1.40745732, ..., -0.49975216,
          1.99570839, -0.49917837],
        [ 1.70055927, -1.53939995, -1.11450787, ..., -0.49975216,
         -0.50107521,  2.00329193],
        ...,
        [-0.36749417, -1.10862268,

## Logistic Regression – From Scratch

### Sigmoid Function/Loss Function (Binary Cross-Entropy)

In [7]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(y, y_hat):
    epsilon = 1e-9
    return -np.mean(y * np.log(y_hat + epsilon) + (1 - y) * np.log(1 - y_hat + epsilon))

### Convert Data to NumPy

In [8]:
X_train_np = X_train_scaled
y_train_np = y_train.values
X_test_np = X_test_scaled
y_test_np = y_test.values

### Initialize Parameters

In [9]:
n_features = X_train_np.shape[1]
weights = np.zeros(n_features)
bias = 0
learning_rate = 0.01
epochs = 1000

### Gradient Descent Training

In [10]:
for epoch in range(epochs):
    linear_output = np.dot(X_train_np, weights) + bias
    y_hat = sigmoid(linear_output)
    dw = (1 / len(y_train_np)) * np.dot(X_train_np.T, (y_hat - y_train_np))
    db = (1 / len(y_train_np)) * np.sum(y_hat - y_train_np)
    weights -= learning_rate * dw
    bias -= learning_rate * db
    if epoch % 100 == 0:
        loss = compute_loss(y_train_np, y_hat)
        print(f"Epoch {epoch}: Loss = {loss}")

Epoch 0: Loss = 0.6931471785599455
Epoch 100: Loss = 0.5703956747594519
Epoch 200: Loss = 0.49462675645281534
Epoch 300: Loss = 0.44619793633296684
Epoch 400: Loss = 0.41401195176017147
Epoch 500: Loss = 0.39182060842556743
Epoch 600: Loss = 0.37601199757171827
Epoch 700: Loss = 0.3644240468986896
Epoch 800: Loss = 0.3557160541291665
Epoch 900: Loss = 0.3490288444427873


### Predict (Scratch Model)

In [11]:
# Prediction Function
def predict(X):
    linear = np.dot(X, weights) + bias
    probs = sigmoid(linear)
    return (probs >= 0.5).astype(int)


In [12]:
y_pred_scratch = predict(X_test_np)

### Accuracy (Scratch Model)

In [13]:
acc_scratch = np.mean(y_pred_scratch == y_test_np)
print("Scratch Logistic Regression Accuracy:", acc_scratch)

Scratch Logistic Regression Accuracy: 0.8838652829449775


## Logistic Regression – Using Sklearn

In [14]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
print("Sklearn Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

Sklearn Logistic Regression Accuracy: 0.6765028392402584


## Random Forest Classifier

In [15]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.8849226551791658
